# RQ2: Mimicry Diagnostic

**Research Question:** What quantitative diagnostic reliably distinguishes authentic holographic wormhole dynamics from non-holographic systems that produce similar-looking transmission correlators?

## Comparison Systems
- **System A**: Free fermion (random quadratic Hamiltonian)
- **System B**: SYK_q with q=2 (free) and q=4 (chaotic)
- **System C**: Mixed-field Ising chain (tunable chaos)
- **System D**: Non-Gaussian SYK (bimodal couplings)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.syk import SYKHamiltonian
from src.comparison_systems import FreeFermionSystem, SYKq, MixedFieldIsing, NonGaussianSYK
from src.observables import level_spacing_ratio, spectral_form_factor, extract_lyapunov
from src.disorder_average import save_results, load_results, check_cache
from src.utils import ensure_dir

ensure_dir('../results')
ensure_dir('../data')

plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14,
    'legend.fontsize': 10, 'figure.dpi': 150,
})

print('RQ2 mimicry diagnostic notebook loaded.')

## 1. Comparison System Diagnostics

Compute level spacing ratios and spectral form factors for each comparison system to establish their spectral character.

In [ ]:
N = 10  # Majorana fermions per system
n_realizations = 30

cache_path = '../data/rq2_diagnostics.npz'

if check_cache(cache_path):
    data = load_results(cache_path)
    print('Loaded RQ2 diagnostics from cache.')
    results = {}
    for name in ['SYK_q4', 'SYK_q2', 'FreeFermion', 'Ising_chaotic', 'Ising_integrable', 'NonGauss_bimodal']:
        results[name] = {
            'r_mean': float(data[f'{name}_r_mean']),
            'r_err': float(data[f'{name}_r_err']),
        }
else:
    results = {}
    
    # Standard SYK q=4
    r_all = []
    for seed in range(n_realizations):
        syk = SYKHamiltonian(N, seed=seed, use_sparse=False)
        evals, _ = syk.diagonalize()
        r_vals, _ = level_spacing_ratio(evals)
        r_all.extend(r_vals)
    results['SYK_q4'] = {'r_mean': np.mean(r_all), 'r_err': np.std(r_all)/np.sqrt(len(r_all))}
    print(f'SYK q=4: <r> = {results["SYK_q4"]["r_mean"]:.4f}')
    
    # SYK q=2
    r_all = []
    for seed in range(n_realizations):
        syk2 = SYKq(N, q=2, seed=seed, use_sparse=False)
        evals, _ = syk2.diagonalize()
        r_vals, _ = level_spacing_ratio(evals)
        r_all.extend(r_vals)
    results['SYK_q2'] = {'r_mean': np.mean(r_all), 'r_err': np.std(r_all)/np.sqrt(len(r_all))}
    print(f'SYK q=2: <r> = {results["SYK_q2"]["r_mean"]:.4f}')
    
    # Free fermion
    r_all = []
    for seed in range(n_realizations):
        ff = FreeFermionSystem(N, seed=seed, use_sparse=False)
        evals, _ = ff.diagonalize()
        r_vals, _ = level_spacing_ratio(evals)
        r_all.extend(r_vals)
    results['FreeFermion'] = {'r_mean': np.mean(r_all), 'r_err': np.std(r_all)/np.sqrt(len(r_all))}
    print(f'Free fermion: <r> = {results["FreeFermion"]["r_mean"]:.4f}')
    
    # Ising chaotic (h_x=0.5, h_z=0.5)
    n_qubits = N // 2
    r_all = []
    ising_c = MixedFieldIsing(n_qubits, J_z=1.0, h_x=0.5, h_z=0.5)
    evals, _ = ising_c.diagonalize()
    r_vals, _ = level_spacing_ratio(evals)
    r_all.extend(r_vals)
    results['Ising_chaotic'] = {'r_mean': np.mean(r_all), 'r_err': np.std(r_all)/np.sqrt(len(r_all))}
    print(f'Ising chaotic: <r> = {results["Ising_chaotic"]["r_mean"]:.4f}')
    
    # Ising integrable (h_z=0)
    r_all = []
    ising_i = MixedFieldIsing(n_qubits, J_z=1.0, h_x=1.0, h_z=0.0)
    evals, _ = ising_i.diagonalize()
    r_vals, _ = level_spacing_ratio(evals)
    r_all.extend(r_vals)
    results['Ising_integrable'] = {'r_mean': np.mean(r_all), 'r_err': np.std(r_all)/np.sqrt(len(r_all))}
    print(f'Ising integrable: <r> = {results["Ising_integrable"]["r_mean"]:.4f}')
    
    # Non-Gaussian SYK (bimodal)
    r_all = []
    for seed in range(n_realizations):
        ng = NonGaussianSYK(N, seed=seed, distribution='bimodal', use_sparse=False)
        evals, _ = ng.diagonalize()
        r_vals, _ = level_spacing_ratio(evals)
        r_all.extend(r_vals)
    results['NonGauss_bimodal'] = {'r_mean': np.mean(r_all), 'r_err': np.std(r_all)/np.sqrt(len(r_all))}
    print(f'Non-Gaussian bimodal: <r> = {results["NonGauss_bimodal"]["r_mean"]:.4f}')
    
    save_data = {}
    for name, vals in results.items():
        save_data[f'{name}_r_mean'] = vals['r_mean']
        save_data[f'{name}_r_err'] = vals['r_err']
    save_results(cache_path, **save_data)
    print('Saved to cache.')

In [ ]:
# Plot comparison of diagnostics
fig, ax = plt.subplots(figsize=(10, 6))

names = list(results.keys())
r_vals = [results[n]['r_mean'] for n in names]
r_errs = [results[n]['r_err'] for n in names]

colors = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c']
x_pos = np.arange(len(names))

bars = ax.bar(x_pos, r_vals, yerr=r_errs, capsize=5, color=colors[:len(names)], alpha=0.8)

ax.axhline(y=0.5307, color='r', linestyle='--', alpha=0.5, label='GOE')
ax.axhline(y=0.6027, color='b', linestyle='--', alpha=0.5, label='GUE')
ax.axhline(y=0.3863, color='gray', linestyle=':', alpha=0.5, label='Poisson')

ax.set_xticks(x_pos)
ax.set_xticklabels(['SYK\nq=4', 'SYK\nq=2', 'Free\nFermion', 'Ising\nchaotic', 'Ising\nintegrable', 'Non-Gauss\nbimodal'], fontsize=10)
ax.set_ylabel(r'$\langle r \rangle$')
ax.set_title(f'Level Spacing Ratio Across Systems (N={N})')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/04_comparison_level_spacing.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Spectral Form Factor Comparison

The spectral form factor reveals whether a system has RMT spectral correlations (dip-ramp-plateau) or not.

In [ ]:
# SFF for each system
t_sff = np.logspace(-1, 3, 300)
n_real_sff = 20

sff_data = {}

# SYK q=4
K_all = []
for seed in range(n_real_sff):
    syk = SYKHamiltonian(N, seed=seed, use_sparse=False)
    evals, _ = syk.diagonalize()
    K_all.append(spectral_form_factor(evals, t_sff))
sff_data['SYK_q4'] = np.mean(K_all, axis=0)

# SYK q=2
K_all = []
for seed in range(n_real_sff):
    syk2 = SYKq(N, q=2, seed=seed, use_sparse=False)
    evals, _ = syk2.diagonalize()
    K_all.append(spectral_form_factor(evals, t_sff))
sff_data['SYK_q2'] = np.mean(K_all, axis=0)

# Free fermion
K_all = []
for seed in range(n_real_sff):
    ff = FreeFermionSystem(N, seed=seed, use_sparse=False)
    evals, _ = ff.diagonalize()
    K_all.append(spectral_form_factor(evals, t_sff))
sff_data['FreeFermion'] = np.mean(K_all, axis=0)

# Non-Gaussian bimodal
K_all = []
for seed in range(n_real_sff):
    ng = NonGaussianSYK(N, seed=seed, distribution='bimodal', use_sparse=False)
    evals, _ = ng.diagonalize()
    K_all.append(spectral_form_factor(evals, t_sff))
sff_data['NonGauss_bimodal'] = np.mean(K_all, axis=0)

print('SFF computation complete.')

In [ ]:
# Plot SFF comparison
fig, ax = plt.subplots(figsize=(10, 6))

dim = 2 ** (N // 2)
style = {
    'SYK_q4': ('SYK q=4 (holographic)', '#2ecc71', '-'),
    'SYK_q2': ('SYK q=2 (free)', '#3498db', '--'),
    'FreeFermion': ('Free fermion', '#e74c3c', ':'),
    'NonGauss_bimodal': ('Non-Gaussian bimodal', '#1abc9c', '-.'),
}

for name, K in sff_data.items():
    label, color, ls = style[name]
    ax.loglog(t_sff, K, color=color, linestyle=ls, linewidth=2, label=label, alpha=0.8)

ax.axhline(y=1.0/dim, color='k', linestyle=':', alpha=0.3, label=f'1/d = 1/{dim}')
ax.set_xlabel('t [1/J]')
ax.set_ylabel('K(t)')
ax.set_title(f'Spectral Form Factor Comparison (N={N})')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(1e-4, 2)

plt.tight_layout()
plt.savefig('../results/04_sff_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Proposed Diagnostic: Combined Spectral-Chaos Score

A proposed diagnostic combining:
1. Level spacing ratio deviation from RMT: $\delta r = |\langle r \rangle - r_{\text{RMT}}| / (r_{\text{RMT}} - r_{\text{Poisson}})$
2. Presence of SFF ramp (spectral rigidity)
3. Lyapunov exponent proximity to chaos bound

The most discriminating single diagnostic is the **spectral form factor ramp slope**, which directly probes RMT spectral correlations that are necessary for holographic behavior.

In [ ]:
# Compute a holographic score for each system
print('=== Holographic Score Summary ===')
print()

r_rmt = 0.5307  # GOE reference
r_poisson = 0.3863

for name in results:
    r_val = results[name]['r_mean']
    
    # Spectral chaos score: 1 = RMT, 0 = Poisson
    chaos_score = max(0, min(1, (r_val - r_poisson) / (r_rmt - r_poisson)))
    
    # SFF ramp score: check if SFF has a ramp (linear growth in log-log)
    if name in sff_data:
        K = sff_data[name]
        # Look for ramp in intermediate time window
        ramp_region = (t_sff > 10) & (t_sff < 500)
        if np.any(ramp_region):
            K_ramp = K[ramp_region]
            t_ramp = t_sff[ramp_region]
            # Check if K is increasing in this region
            slope = np.polyfit(np.log10(t_ramp), np.log10(K_ramp + 1e-20), 1)[0]
            ramp_score = max(0, min(1, slope))  # positive slope = ramp
        else:
            ramp_score = 0
    else:
        ramp_score = 0
    
    # Combined score
    combined = 0.5 * chaos_score + 0.5 * ramp_score
    
    print(f'{name:20s}: chaos={chaos_score:.3f}, ramp={ramp_score:.3f}, combined={combined:.3f}')

print()
print('A holographic system should have combined score close to 1.')
print('Non-holographic systems should have scores significantly below 1.')

## 4. RQ2 Findings Summary

The combined spectral-chaos diagnostic provides a quantitative way to distinguish holographic from non-holographic systems:

1. **SYK q=4** (holographic): Both level spacing ratio and SFF ramp present. High holographic score.
2. **SYK q=2** (free): Poisson-like statistics, no ramp. Low holographic score.
3. **Free fermion**: Integrable, no chaos. Low holographic score.
4. **Ising chaotic**: May have chaos but different spectral structure. Intermediate score.
5. **Non-Gaussian SYK**: May retain chaos but differ in finer spectral correlations.

The key finding: **spectral rigidity** (SFF ramp) is the most discriminating single diagnostic, since it probes the specific spectral correlations that underlie the holographic dual.